In [36]:
import torch
from  torch.utils.data import Dataset, DataLoader
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo utilizado:", device)

Dispositivo utilizado: cpu


In [37]:
data = pd.read_csv(
    "australian.dat",
    header=None,
    sep=r"\s+"
)

print("Dimensiones del dataset:")
print(data.shape)
data.head()

Dimensiones del dataset:
(690, 15)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1,22.08,11.46,2,4,4,1.585,0,0,0,1,2,100,1213,0
1,0,22.67,7.00,2,8,4,0.165,0,0,0,0,2,160,1,0
2,0,29.58,1.75,1,4,4,1.250,0,0,0,1,2,280,1,0
3,0,21.67,11.50,1,5,3,0.000,1,1,11,1,2,0,1,1
4,1,20.17,8.17,2,6,4,1.960,1,1,14,0,2,60,159,1


In [38]:
print(data.info())


print(data.describe())

<class 'pandas.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 15 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       690 non-null    int64  
 1   1       690 non-null    float64
 2   2       690 non-null    float64
 3   3       690 non-null    int64  
 4   4       690 non-null    int64  
 5   5       690 non-null    int64  
 6   6       690 non-null    float64
 7   7       690 non-null    int64  
 8   8       690 non-null    int64  
 9   9       690 non-null    int64  
 10  10      690 non-null    int64  
 11  11      690 non-null    int64  
 12  12      690 non-null    int64  
 13  13      690 non-null    int64  
 14  14      690 non-null    int64  
dtypes: float64(3), int64(12)
memory usage: 81.0 KB
None
               0           1           2           3           4           5   \
count  690.000000  690.000000  690.000000  690.000000  690.000000  690.000000   
mean     0.678261   31.568203    4.758725    1.766667    7

In [39]:
X =data.iloc[:,:-1].values
y = data[14].values
print(X.shape)
print(y.shape)

(690, 14)
(690,)


In [40]:
def normalize(X):
    mu=np.mean(X, axis=0)
    sigma= np.std(X, axis=0)
    X_norm= (X-mu)/sigma
    return X_norm, mu, sigma
X_norm, mu, sigma =normalize(X)

In [47]:
split=int(len(X_norm)*0.75)
X_train = X_norm[:split]
y_train = y[:split]
X_test = X_norm[split:]
y_test = y[split:]
print("Entrenamiento:", X_train.shape)
print("Test:", X_test.shape)

Entrenamiento: (517, 14)
Test: (173, 14)


In [48]:
class australian(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, ix):
        return self.X[ix], self.y[ix]

train_dataset = australian(X_train, y_train)
test_dataset  = australian(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

In [49]:
class RegresionLogistica(torch.nn.Module):
    def __init__(self, input_size):
        super(RegresionLogistica, self).__init__()
        self.fc = torch.nn.Linear(input_size, 1)
        self.sigmoid = torch.nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(x))

model = RegresionLogistica(X_train.shape[1]).to(device)

In [51]:
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

epochs = 100
best_acc = 0
PATH = "./checkpoint_australia.pt"
cost_history = []

for epoch in range(1, epochs + 1):
    model.train()
    losses = []
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_b), y_b)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    cost_history.append(np.mean(losses))

    # checkpoint
    model.eval()
    with torch.no_grad():
        preds = (model(torch.from_numpy(X_test).to(device)) >= 0.5).float().cpu().numpy()
    acc = np.mean(preds.flatten() == y_test)
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), PATH)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{epochs}  loss: {np.mean(cost_history):.4f}  acc: {acc*100:.2f}%")

model.load_state_dict(torch.load(PATH))
print(f"\nMejor accuracy: {best_acc*100:.2f}%")

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float

In [ ]:
plt.plot(cost_history)
plt.title("Evolución de la función de costo")
plt.xlabel("Iteraciones")
plt.ylabel("Costo")
plt.show()